In [ ]:
# 📦 Essential Imports
import sys
import os
import time
import warnings
from pathlib import Path

# Add project root to path
project_root = Path("/workspaces/football_analysis")
sys.path.append(str(project_root))

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# ML libraries with error handling
try:
    from sklearn.metrics import silhouette_score, adjusted_rand_score
    from sklearn.decomposition import PCA
    print("✅ Scikit-learn available")
except ImportError:
    print("⚠️ Scikit-learn not available - some metrics will be limited")

try:
    import umap
    print("✅ UMAP available")
except ImportError:
    print("⚠️ UMAP not available - will use PCA for dimensionality reduction")
    umap = None

# SigLIP Team Assignment Processor
try:
    from football_ai.assignment.team_assignment_processor import SigLIPTeamAssignmentProcessor
    from football_ai.domain.data_models import VideoData, FrameData, Detection, BoundingBox, ObjectType
    print("✅ SigLIP processor imports successful")
except ImportError as e:
    print(f"❌ Failed to import SigLIP processor: {e}")
    print("🔧 Please check if the football_ai module is properly installed")

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print(f"\n📋 IMPORT SUMMARY:")
print(f"📁 Project root: {project_root}")
print(f"🖥️  Available processors: SigLIPTeamAssignmentProcessor")
print(f"🎨 Visualization mode: Matplotlib only")
print("✅ Setup complete!")

In [ ]:
# 🎬 Real Video Data Loader with GPU Support
import cv2
import torch
from football_ai.detection.object_detection_processor import ObjectDetectionProcessor

def load_real_video_data(video_path=None, max_frames=50, use_gpu=True):
    """Load real video data using the project's detection pipeline with GPU support."""
    if video_path is None:
        input_dir = project_root / "input_videos"
        video_files = [f for f in input_dir.glob("*.mp4") if f.name != "put_input_video_here.txt"]
        if not video_files:
            raise FileNotFoundError("⚠️ No video files found in input_videos directory. Please add a video file.")
        video_path = video_files[0]
    
    print(f"🎬 Loading real video data from: {video_path.name}")
    print(f"   📊 Max frames to process: {max_frames}")
    
    # Check device availability for YOLO
    device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
    print(f"🖥️ Using device for YOLO detection: {device}")
    if device == "cuda":
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🎮 GPU: {gpu_name}")
    
    # Initialize object detector - this must work for real evaluation
    try:
        model_path = project_root / "models" / "detect" / "best.pt"
        if not model_path.exists():
            raise FileNotFoundError(f"⚠️ Model file not found: {model_path}")
        
        print(f"🔄 Loading YOLO model on {device}...")
        detector = ObjectDetectionProcessor(
            model_path=str(model_path),
            confidence_threshold=0.3
        )
        
        # Move YOLO model to GPU if available
        if hasattr(detector, 'model') and hasattr(detector.model, 'to'):
            detector.model.to(device)
            print(f"✅ YOLO model moved to {device}")
        
        print("✅ Object detector loaded successfully")
    except Exception as e:
        print(f"❌ Failed to load object detector: {e}")
        if device == "cuda":
            print("🔄 Retrying with CPU...")
            try:
                detector = ObjectDetectionProcessor(
                    model_path=str(model_path),
                    confidence_threshold=0.3
                )
                print("✅ Object detector loaded successfully on CPU")
                device = "cpu"  # Update device for logging
            except Exception as cpu_error:
                raise RuntimeError(f"❌ Failed to load object detector on both GPU and CPU: {cpu_error}")
        else:
            raise RuntimeError(f"❌ Failed to load object detector: {e}")
    
    # Load video
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"❌ Could not open video file: {video_path}")
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"📺 Video properties:")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Total frames: {total_frames}")
    
    frames = []
    frame_idx = 0
    processed_count = 0
    
    print(f"🔄 Processing frames on {device}...")
    
    while processed_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Create frame data
        timestamp = frame_idx / fps
        
        # Run real object detection
        try:
            detections = detector._detect_objects(frame)
            if processed_count % 10 == 0:
                print(f"🎯 Frame {frame_idx}: {len(detections)} detections")
        except Exception as e:
            print(f"⚠️ Detection failed for frame {frame_idx}: {e}")
            # Skip this frame rather than creating mock data
            frame_idx += 1
            continue
        
        frame_data = FrameData(
            frame_number=frame_idx,
            timestamp=timestamp,
            raw_frame=frame,
            detections=detections,
            metadata={
                'source': 'real_video', 
                'detector': 'yolo', 
                'device': device,
                'model_path': str(model_path)
            }
        )
        frames.append(frame_data)
        
        if processed_count % 10 == 0:
            detection_count = len(detections)
            print(f"📊 Processed frame {processed_count}/{max_frames}, detections: {detection_count}")
        
        frame_idx += 1
        processed_count += 1
    
    cap.release()
    
    if not frames:
        raise RuntimeError("❌ No frames were successfully processed")
    
    # Create VideoData with correct constructor parameters
    video_data = VideoData(
        video_path=str(video_path),
        frame_rate=fps,
        resolution=(width, height),
        duration=len(frames) / fps,
        frames=frames,
        metadata={
            'source_file': str(video_path),
            'total_frames_available': total_frames,
            'frames_processed': len(frames),
            'detector_used': 'real_yolo',
            'detection_model': str(model_path),
            'device_used': device
        }
    )
    
    total_detections = sum(len(f.detections) for f in frames)
    players_count = sum(1 for f in frames for d in f.detections if d.object_type in ['player', 'goalkeeper'])
    ball_count = sum(1 for f in frames for d in f.detections if d.object_type == 'ball')
    referee_count = sum(1 for f in frames for d in f.detections if d.object_type == 'referee')
    
    print(f"✅ Real video data loaded successfully!")
    print(f"   📈 Total frames: {len(frames)}")
    print(f"   🎯 Total detections: {total_detections}")
    print(f"   👥 Player detections: {players_count}")
    print(f"   ⚽ Ball detections: {ball_count}")
    print(f"   👔 Referee detections: {referee_count}")
    print(f"   ⏱️ Duration: {video_data.duration:.2f}s")
    print(f"   🔧 Detection method: Real YOLO model on {device}")
    
    return video_data

# Load real test data with GPU support
print("🚀 Loading real video data with GPU acceleration...")
try:
    test_video_data = load_real_video_data(max_frames=30, use_gpu=True)
except Exception as e:
    print(f"❌ Failed to load real video data: {e}")
    print("💡 Please ensure:")
    print("   1. Video files are present in input_videos/ directory")
    print("   2. YOLO model (best.pt) is available in models/detect/")
    print("   3. Video files are in a supported format (mp4)")
    print("   4. Sufficient GPU/CPU memory available")
    test_video_data = None

In [ ]:
# 🤖 SigLIP Processor Initialization with GPU Detection
import torch

def detect_best_device():
    """Detect the best available device (GPU or CPU)"""
    if torch.cuda.is_available():
        device = "cuda"
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"🎮 GPU detected: {gpu_name}")
        print(f"💾 GPU memory: {gpu_memory:.1f}GB")
        return device
    else:
        print("🖥️ No GPU detected, using CPU")
        return "cpu"

def create_siglip_processor(device=None, batch_size=8, n_clusters=2):
    """Create SigLIP processor with automatic device detection"""
    
    # Auto-detect best device if not specified
    if device is None:
        device = detect_best_device()
    
    print("🤖 Initializing SigLIP Team Assignment Processor...")
    print(f"   🖥️  Device: {device}")
    print(f"   📦 Batch size: {batch_size}")
    print(f"   🎯 Number of clusters: {n_clusters}")
    
    try:
        processor = SigLIPTeamAssignmentProcessor(
            device=device,
            batch_size=batch_size,
            n_clusters=n_clusters
        )
        print("✅ SigLIP processor created successfully!")
        print("✅ Real SigLIP model loaded and ready for evaluation")
        return processor, True
        
    except Exception as e:
        print(f"❌ SigLIP processor initialization failed: {e}")
        print(f"🔍 Error type: {type(e).__name__}")
        
        # Check common issues
        if "Failed to load SigLIP model" in str(e):
            print("💡 SigLIP model loading failed. Possible causes:")
            print("   - Internet connection required for first-time model download")
            print("   - Insufficient memory (need ~2GB+ for SigLIP)")
            print("   - HuggingFace transformers/model loading issue")
        elif "CUDA" in str(e) and device == "cuda":
            print("💡 GPU-related error detected. Trying CPU fallback...")
            try:
                print("🔄 Retrying with CPU...")
                processor = SigLIPTeamAssignmentProcessor(
                    device="cpu",
                    batch_size=batch_size,
                    n_clusters=n_clusters
                )
                print("✅ SigLIP processor created successfully on CPU!")
                return processor, True
            except Exception as cpu_error:
                print(f"❌ CPU fallback also failed: {cpu_error}")
        elif "transformers" in str(e):
            print("💡 Transformers library issue. Try:")
            print("   pip install --upgrade transformers torch")
        elif "memory" in str(e).lower() or "oom" in str(e).lower():
            print("💡 Memory issue. Try:")
            print("   - Reduce batch_size (current: {})".format(batch_size))
            print("   - Close other applications")
            print("   - Use CPU instead of GPU")
        
        print("🔧 Debug info:")
        print(f"   Torch version: {torch.__version__}")
        print(f"   CUDA available: {torch.cuda.is_available()}")
        if torch.cuda.is_available():
            print(f"   CUDA version: {torch.version.cuda}")
            print(f"   GPU count: {torch.cuda.device_count()}")
        
        return None, False

# Initialize processor with auto device detection
try:
    print("🔄 Attempting to load real SigLIP model...")
    print("🔍 Detecting best available device...")
    
    # Try with auto-detected device first
    siglip_processor, model_available = create_siglip_processor()
    
    if model_available:
        print("🎉 Ready for real SigLIP evaluation!")
    else:
        print("❌ SigLIP model initialization failed")
        
        # Try with reduced batch size as fallback
        print("🔄 Trying with reduced batch size...")
        siglip_processor, model_available = create_siglip_processor(batch_size=4)
        
        if model_available:
            print("🎉 Ready for real SigLIP evaluation with reduced batch size!")
        else:
            print("❌ All SigLIP initialization attempts failed")
            
except Exception as e:
    print(f"❌ Unexpected error during initialization: {e}")
    siglip_processor = None
    model_available = False

# Final status
if siglip_processor is not None:
    print(f"✅ SigLIP processor ready: {type(siglip_processor)}")
else:
    print("❌ SigLIP processor not available")
    print("💡 Check the error messages above for troubleshooting steps")

In [ ]:
# 🔍 Detailed SigLIP Error Diagnosis
import traceback

print("🔍 Testing SigLIP initialization step by step...")

# Test 1: Check if transformers is working
try:
    from transformers import AutoProcessor, SiglipVisionModel
    print("✅ Transformers import successful")
except Exception as e:
    print(f"❌ Transformers import failed: {e}")
    print(traceback.format_exc())

# Test 2: Check internet connectivity by testing a simple model load
try:
    print("🌐 Testing model download capability...")
    from transformers import AutoProcessor
    
    # Try to load the processor (smaller download first)
    print("📥 Attempting to load SigLIP processor...")
    processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
    print("✅ SigLIP processor loaded successfully")
    
    # Try to load the model
    print("📥 Attempting to load SigLIP model...")
    model = SiglipVisionModel.from_pretrained("google/siglip-base-patch16-224")
    print("✅ SigLIP model loaded successfully")
    
    # Test device placement
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Moving model to {device}...")
    model = model.to(device)
    print(f"✅ Model moved to {device} successfully")
    
    print("🎉 All SigLIP components working - processor initialization should succeed!")
    
except Exception as e:
    print(f"❌ SigLIP component test failed: {e}")
    print("📋 Full error traceback:")
    print(traceback.format_exc())
    
    # Check common issues
    if "ConnectionError" in str(e) or "URLError" in str(e):
        print("🌐 Network issue detected - check internet connection")
    elif "HTTPError" in str(e):
        print("🔄 HTTP error - model server may be temporarily unavailable")
    elif "OSError" in str(e) and "disk" in str(e).lower():
        print("💾 Disk space issue - check available storage")
    elif "CUDA" in str(e):
        print("🎮 GPU issue - will fallback to CPU")

# Test 3: Try direct processor initialization
try:
    print("\n🤖 Testing direct SigLIPTeamAssignmentProcessor initialization...")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"🖥️ Using device: {device}")
    
    processor = SigLIPTeamAssignmentProcessor(
        device=device,
        batch_size=4,  # Small batch size
        n_clusters=2
    )
    print("✅ SigLIPTeamAssignmentProcessor created successfully!")
    
except Exception as e:
    print(f"❌ SigLIPTeamAssignmentProcessor failed: {e}")
    print("📋 Full error traceback:")
    print(traceback.format_exc())

In [ ]:
# 📊 Visual Data Exploration
def visualize_video_data(video_data):
    """Create comprehensive visualizations of the loaded video data using matplotlib"""
    
    if video_data is None:
        print("❌ No video data to visualize")
        return
    
    # Extract basic statistics
    frames = video_data.frames
    total_frames = len(frames)
    detections_per_frame = [len(frame.detections) for frame in frames]
    timestamps = [frame.timestamp for frame in frames]
    
    print(f"📈 VIDEO DATA ANALYSIS")
    print("=" * 50)
    print(f"🎬 Total frames: {total_frames}")
    print(f"👥 Total detections: {sum(detections_per_frame)}")
    print(f"📊 Avg detections per frame: {np.mean(detections_per_frame):.1f}")
    print(f"⏱️  Duration: {max(timestamps):.1f} seconds")
    
    # Collect confidence data
    all_confidences = []
    for frame in frames:
        for det in frame.detections:
            all_confidences.append(det.confidence)
    
    # Create matplotlib dashboard
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('🎬 Video Data Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # 1. Detections over time
    axes[0, 0].plot(timestamps, detections_per_frame, 'o-', color='#3498db', linewidth=2, markersize=4)
    axes[0, 0].set_title('Detections Over Time')
    axes[0, 0].set_xlabel('Time (seconds)')
    axes[0, 0].set_ylabel('Number of Detections')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Detection count histogram
    axes[0, 1].hist(detections_per_frame, bins=10, alpha=0.7, color='#e74c3c', edgecolor='black')
    axes[0, 1].set_title('Detection Count Distribution')
    axes[0, 1].set_xlabel('Detections per Frame')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Sample frame detection areas
    if frames and frames[0].detections:
        sample_frame = frames[len(frames)//2]
        detection_areas = []
        confidences = []
        
        for det in sample_frame.detections:
            bbox = det.bbox
            area = (bbox.x2 - bbox.x1) * (bbox.y2 - bbox.y1)
            detection_areas.append(area)
            confidences.append(det.confidence)
        
        scatter = axes[1, 0].scatter(range(len(detection_areas)), detection_areas, 
                                   c=confidences, cmap='viridis', s=60, alpha=0.7)
        axes[1, 0].set_title('Sample Frame Detection Areas')
        axes[1, 0].set_xlabel('Detection Index')
        axes[1, 0].set_ylabel('Bounding Box Area')
        axes[1, 0].grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=axes[1, 0], label='Confidence')
    else:
        axes[1, 0].text(0.5, 0.5, 'No detections available', ha='center', va='center', 
                      transform=axes[1, 0].transAxes, fontsize=12)
        axes[1, 0].set_title('Sample Frame Detection Areas')
    
    # 4. Confidence distribution
    if all_confidences:
        axes[1, 1].hist(all_confidences, bins=20, alpha=0.7, color='#2ecc71', edgecolor='black')
        axes[1, 1].set_title('Detection Confidence Distribution')
        axes[1, 1].set_xlabel('Confidence Score')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'No confidence data', ha='center', va='center', 
                      transform=axes[1, 1].transAxes, fontsize=12)
        axes[1, 1].set_title('Detection Confidence Distribution')
    
    plt.tight_layout()
    plt.show()
    
    return {
        'total_frames': total_frames,
        'total_detections': sum(detections_per_frame),
        'avg_detections_per_frame': np.mean(detections_per_frame),
        'confidence_stats': {
            'mean': np.mean(all_confidences) if all_confidences else 0,
            'std': np.std(all_confidences) if all_confidences else 0,
            'min': np.min(all_confidences) if all_confidences else 0,
            'max': np.max(all_confidences) if all_confidences else 0
        }
    }

# Visualize the loaded data
print("🎨 Creating data visualization dashboard...")
data_stats = visualize_video_data(test_video_data)

In [ ]:
# 🎯 Team Assignment Processing & Evaluation
def run_team_assignment_evaluation(processor, video_data):
    """Run comprehensive team assignment evaluation with real SigLIP processing"""
    
    if video_data is None:
        print("❌ No video data available for evaluation")
        return None
    
    if processor is None:
        print("❌ No SigLIP processor available for evaluation")
        return None
    
    print("🚀 TEAM ASSIGNMENT EVALUATION")
    print("=" * 50)
    
    # Check if we have player detections
    total_players = sum(len([d for d in frame.detections if d.object_type == ObjectType.PLAYER]) 
                       for frame in video_data.frames)
    
    if total_players == 0:
        print("⚠️ No player detections found in video data")
        return None
    
    print(f"👥 Total players detected: {total_players}")
    print(f"🎬 Frames to process: {len(video_data.frames)}")
    
    # Real SigLIP Processing only
    try:
        print("\n🤖 Running SigLIP-based team assignment...")
        
        # Train the processor
        print("🔄 Training SigLIP processor...")
        start_time = time.time()
        processor.train(video_data)
        training_time = time.time() - start_time
        print(f"✅ Training completed in {training_time:.2f} seconds")
        
        # Process the video
        print("🔄 Processing video with trained model...")
        start_time = time.time()
        processed_video = processor.process(video_data)
        processing_time = time.time() - start_time
        print(f"✅ Processing completed in {processing_time:.2f} seconds")
        
        # Analyze results
        team_assignments = {}
        for frame in processed_video.frames:
            for detection in frame.detections:
                if (detection.object_type == ObjectType.PLAYER and 
                    detection.metadata and 'team_id' in detection.metadata):
                    team_id = detection.metadata['team_id']
                    team_assignments[team_id] = team_assignments.get(team_id, 0) + 1
        
        print(f"📊 Team assignment results:")
        for team_id, count in team_assignments.items():
            print(f"   Team {team_id}: {count} assignments")
        
        results = {
            'siglip_results': {
                'training_time': training_time,
                'processing_time': processing_time,
                'processed_video': processed_video,
                'method': 'SigLIP',
                'team_assignments': team_assignments
            }
        }
        
        return results
        
    except Exception as e:
        print(f"❌ SigLIP processing failed: {e}")
        print("💡 This could be due to:")
        print("   - SigLIP model not properly loaded")
        print("   - Insufficient player detections for clustering")
        print("   - Network connectivity issues (for model download)")
        raise e

# Run the evaluation with real data only
if test_video_data is not None and siglip_processor is not None:
    print("🎯 Starting real SigLIP evaluation...")
    evaluation_results = run_team_assignment_evaluation(siglip_processor, test_video_data)
elif test_video_data is None:
    print("❌ Cannot run evaluation: No video data loaded")
    evaluation_results = None
elif siglip_processor is None:
    print("❌ Cannot run evaluation: SigLIP processor not available")
    evaluation_results = None
else:
    print("❌ Cannot run evaluation: Missing requirements")
    evaluation_results = None

In [ ]:
# 📈 Results Visualization Dashboard
def create_results_dashboard(evaluation_results):
    """Create comprehensive visualization of real SigLIP team assignment results"""
    
    if evaluation_results is None:
        print("❌ No evaluation results to visualize")
        return None
    
    # Use real SigLIP results only
    if 'siglip_results' not in evaluation_results:
        print("❌ No SigLIP results found - this evaluation requires real processing")
        return None
        
    results = evaluation_results['siglip_results']
    method_name = "SigLIP"
    processed_video = results['processed_video']
    
    print(f"📊 REAL SIGLIP RESULTS ANALYSIS")
    print("=" * 60)
    
    # Extract team assignment data
    frame_data = []
    team_counts = {0: 0, 1: 0}
    assignment_timeline = []
    confidence_scores = []
    
    for frame in processed_video.frames:
        frame_teams = {0: 0, 1: 0}
        frame_assignments = []
        
        for detection in frame.detections:
            if (detection.object_type == ObjectType.PLAYER and 
                detection.metadata and 'team_id' in detection.metadata):
                
                team_id = detection.metadata['team_id']
                frame_teams[team_id] += 1
                team_counts[team_id] += 1
                
                # Extract confidence if available
                confidence = detection.metadata.get('assignment_confidence', detection.confidence)
                confidence_scores.append(confidence)
                
                frame_assignments.append({
                    'frame': frame.frame_number,
                    'team_id': team_id,
                    'confidence': confidence,
                    'bbox_area': (detection.bbox.x2 - detection.bbox.x1) * 
                                (detection.bbox.y2 - detection.bbox.y1)
                })
        
        frame_data.append({
            'frame': frame.frame_number,
            'timestamp': frame.timestamp,
            'team_0_count': frame_teams[0],
            'team_1_count': frame_teams[1],
            'total_players': frame_teams[0] + frame_teams[1]
        })
        
        assignment_timeline.extend(frame_assignments)
    
    df_frames = pd.DataFrame(frame_data)
    df_assignments = pd.DataFrame(assignment_timeline)
    
    # Print summary statistics
    total_team_players = team_counts[0] + team_counts[1]
    if total_team_players > 0:
        print(f"⚽ Team 0 players: {team_counts[0]} ({team_counts[0]/total_team_players*100:.1f}%)")
        print(f"🔵 Team 1 players: {team_counts[1]} ({team_counts[1]/total_team_players*100:.1f}%)")
        balance_ratio = min(team_counts.values()) / max(team_counts.values()) if max(team_counts.values()) > 0 else 0
        print(f"⚖️  Team balance ratio: {balance_ratio:.3f}")
    
    print(f"⏱️  Training time: {results['training_time']:.2f}s")
    print(f"🚀 Processing time: {results['processing_time']:.2f}s")
    
    if confidence_scores:
        avg_confidence = np.mean(confidence_scores)
        print(f"🎯 Average assignment confidence: {avg_confidence:.3f}")
    
    # Create visualizations
    if not df_frames.empty:
        # Create matplotlib dashboard for real results
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
        fig.suptitle('🤖 Real SigLIP Team Assignment Analysis Dashboard', fontsize=16, fontweight='bold')
        
        # 1. Team assignment timeline
        axes[0, 0].plot(df_frames['timestamp'], df_frames['team_0_count'], 'o-', 
                       color='#e74c3c', linewidth=2, markersize=4, label='Team 0')
        axes[0, 0].plot(df_frames['timestamp'], df_frames['team_1_count'], 'o-', 
                       color='#3498db', linewidth=2, markersize=4, label='Team 1')
        axes[0, 0].set_title('Team Assignment Over Time')
        axes[0, 0].set_xlabel('Time (seconds)')
        axes[0, 0].set_ylabel('Player Count')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # 2. Team distribution pie chart
        if total_team_players > 0:
            colors = ['#e74c3c', '#3498db']
            wedges, texts, autotexts = axes[0, 1].pie([team_counts[0], team_counts[1]], 
                                                     labels=['Team 0', 'Team 1'], 
                                                     colors=colors, autopct='%1.1f%%',
                                                     startangle=90)
            axes[0, 1].set_title('Real SigLIP Team Distribution')
        else:
            axes[0, 1].text(0.5, 0.5, 'No team assignments', ha='center', va='center', 
                           transform=axes[0, 1].transAxes)
            axes[0, 1].set_title('Team Distribution')
        
        # 3. Total players per frame
        axes[1, 0].plot(df_frames['timestamp'], df_frames['total_players'], 'o-', 
                       color='#2ecc71', linewidth=2, markersize=4)
        axes[1, 0].fill_between(df_frames['timestamp'], df_frames['total_players'], alpha=0.3, color='#2ecc71')
        axes[1, 0].set_title('Player Count Per Frame')
        axes[1, 0].set_xlabel('Time (seconds)')
        axes[1, 0].set_ylabel('Total Players')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 4. Assignment confidence distribution
        if confidence_scores:
            axes[1, 1].hist(confidence_scores, bins=15, alpha=0.7, 
                           color='#f39c12', edgecolor='black')
            axes[1, 1].set_title('SigLIP Assignment Confidence')
            axes[1, 1].set_xlabel('Confidence Score')
            axes[1, 1].set_ylabel('Frequency')
            axes[1, 1].grid(True, alpha=0.3)
        else:
            axes[1, 1].text(0.5, 0.5, 'No confidence data', ha='center', va='center', 
                           transform=axes[1, 1].transAxes)
            axes[1, 1].set_title('Assignment Confidence')
        
        # 5. Processing performance
        performance_metrics = ['Training', 'Processing']
        performance_times = [results['training_time'], results['processing_time']]
        colors = ['#9b59b6', '#1abc9c']
        
        bars = axes[2, 0].bar(performance_metrics, performance_times, color=colors, alpha=0.7)
        axes[2, 0].set_title('Real SigLIP Performance')
        axes[2, 0].set_ylabel('Time (seconds)')
        axes[2, 0].grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar, time_val in zip(bars, performance_times):
            height = bar.get_height()
            axes[2, 0].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                           f'{time_val:.2f}s', ha='center', va='bottom')
        
        # 6. Feature quality assessment
        if not df_assignments.empty:
            team_0_areas = df_assignments[df_assignments['team_id'] == 0]['bbox_area']
            team_1_areas = df_assignments[df_assignments['team_id'] == 1]['bbox_area']
            
            box_data = []
            labels = []
            if len(team_0_areas) > 0:
                box_data.append(team_0_areas)
                labels.append('Team 0')
            if len(team_1_areas) > 0:
                box_data.append(team_1_areas)
                labels.append('Team 1')
            
            if box_data:
                bp = axes[2, 1].boxplot(box_data, labels=labels, patch_artist=True)
                colors = ['#e74c3c', '#3498db']
                for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
                    patch.set_facecolor(color)
                    patch.set_alpha(0.7)
                axes[2, 1].set_title('Player Size Distribution by Team')
                axes[2, 1].set_ylabel('Bounding Box Area')
                axes[2, 1].grid(True, alpha=0.3)
            else:
                axes[2, 1].text(0.5, 0.5, 'No bbox data', ha='center', va='center', 
                               transform=axes[2, 1].transAxes)
                axes[2, 1].set_title('Player Size Distribution')
        else:
            axes[2, 1].text(0.5, 0.5, 'No assignment data', ha='center', va='center', 
                           transform=axes[2, 1].transAxes)
            axes[2, 1].set_title('Player Size Distribution')
        
        plt.tight_layout()
        plt.show()
    
    # Return summary statistics
    return {
        'method': method_name,
        'total_assignments': len(df_assignments),
        'team_distribution': team_counts,
        'processing_performance': {
            'training_time': results['training_time'],
            'processing_time': results['processing_time'],
            'fps_estimate': len(processed_video.frames) / results['processing_time'] if results['processing_time'] > 0 else 0
        },
        'quality_metrics': {
            'avg_confidence': np.mean(confidence_scores) if confidence_scores else 0,
            'assignment_consistency': df_frames['total_players'].std() / df_frames['total_players'].mean() if df_frames['total_players'].mean() > 0 else 0,
            'team_balance_ratio': min(team_counts.values()) / max(team_counts.values()) if max(team_counts.values()) > 0 else 0
        }
    }

# Create the results dashboard for real data only
if evaluation_results is not None:
    print("🎨 Creating real SigLIP results dashboard...")
    dashboard_stats = create_results_dashboard(evaluation_results)
else:
    print("❌ No evaluation results available for visualization")
    dashboard_stats = None

# Initialize SigLIP processor with local model
print("🚀 Initializing SigLIP Team Assignment Processor...")

# Use local model path
local_model_path = project_root / "models" / "embed" / "team_assignment_model"

try:
    # Check if local model exists
    if local_model_path.exists():
        print(f"✅ Found local SigLIP model at: {local_model_path}")
        
        # Initialize with local model path and GPU if available
        siglip_processor = SigLIPTeamAssignmentProcessor(
            model_path=str(local_model_path),
            device=device,
            batch_size=32 if device == "cuda" else 16,  # Larger batch on GPU
            n_clusters=2
        )
        
        print(f"✅ SigLIP processor initialized successfully!")
        print(f"   - Model loaded from: {local_model_path}")
        print(f"   - Device: {device}")
        print(f"   - Batch size: {siglip_processor.batch_size}")
        
        model_available = True
        
    else:
        print(f"❌ Local SigLIP model not found at: {local_model_path}")
        print("   Please ensure the model has been downloaded to the models folder.")
        siglip_processor = None
        model_available = False
        
except Exception as e:
    print(f"❌ Failed to initialize SigLIP processor: {e}")
    print(f"   Error type: {type(e).__name__}")
    
    # Try with smaller batch size on GPU in case of memory issues
    if device == "cuda" and "memory" in str(e).lower():
        try:
            print("🔄 Retrying with smaller batch size...")
            siglip_processor = SigLIPTeamAssignmentProcessor(
                model_path=str(local_model_path) if local_model_path.exists() else None,
                device=device,
                batch_size=8,  # Smaller batch for GPU memory issues
                n_clusters=2
            )
            print("✅ SigLIP processor initialized with smaller batch size!")
            model_available = True
        except Exception as e2:
            print(f"❌ Still failed with smaller batch: {e2}")
            siglip_processor = None
            model_available = False
    else:
        siglip_processor = None
        model_available = False

print(f"\n📊 SigLIP Processor Status: {'Available' if model_available else 'Not Available'}")

In [ ]:
# 🎯 Real SigLIP Evaluation Summary & Conclusions
def print_real_evaluation_summary(dashboard_stats, data_stats):
    """Print comprehensive evaluation summary for real SigLIP processing"""
    
    print("🏆 REAL SIGLIP TEAM ASSIGNMENT PROCESSOR - EVALUATION SUMMARY")
    print("=" * 70)
    
    if dashboard_stats and dashboard_stats['method'] == 'SigLIP':
        print(f"\n🔬 METHOD USED: Real SigLIP with Visual Features")
        
        print(f"\n📊 ASSIGNMENT STATISTICS:")
        print(f"   👥 Total player assignments: {dashboard_stats['total_assignments']}")
        
        team_dist = dashboard_stats['team_distribution']
        total_players = sum(team_dist.values())
        if total_players > 0:
            print(f"   ⚽ Team 0: {team_dist[0]} players ({team_dist[0]/total_players*100:.1f}%)")
            print(f"   🔵 Team 1: {team_dist[1]} players ({team_dist[1]/total_players*100:.1f}%)")
            
            # Team balance assessment
            balance_ratio = dashboard_stats['quality_metrics']['team_balance_ratio']
            print(f"   ⚖️  Team balance ratio: {balance_ratio:.3f}")
            
            if balance_ratio > 0.8:
                print("   ✅ Teams are well balanced")
            elif balance_ratio > 0.6:
                print("   ⚠️ Teams are moderately balanced")
            else:
                print("   ❌ Teams are imbalanced - may need hyperparameter tuning")
        
        print(f"\n⚡ REAL-TIME PERFORMANCE:")
        perf = dashboard_stats['processing_performance']
        print(f"   🏃 Training time: {perf['training_time']:.2f} seconds")
        print(f"   🚀 Processing time: {perf['processing_time']:.2f} seconds")
        print(f"   🎬 Estimated FPS: {perf['fps_estimate']:.1f}")
        
        if perf['fps_estimate'] > 20:
            print("   ✅ Real-time capable (>20 FPS)")
        elif perf['fps_estimate'] > 10:
            print("   ⚠️ Near real-time (10-20 FPS)")
        else:
            print("   ❌ Below real-time (<10 FPS) - consider optimization")
        
        print(f"\n🎯 SIGLIP QUALITY ASSESSMENT:")
        quality = dashboard_stats['quality_metrics']
        print(f"   📊 Average assignment confidence: {quality['avg_confidence']:.3f}")
        print(f"   🔄 Assignment consistency: {1-quality['assignment_consistency']:.3f}")
        
        if quality['avg_confidence'] > 0.8:
            print("   ✅ High confidence SigLIP assignments")
        elif quality['avg_confidence'] > 0.6:
            print("   ⚠️ Moderate confidence assignments")
        else:
            print("   ❌ Low confidence assignments - check feature extraction")
        
        print(f"\n📹 VIDEO DATA SUMMARY:")
        if data_stats:
            print(f"   🎬 Frames processed: {data_stats['total_frames']}")
            print(f"   👥 Total detections: {data_stats['total_detections']}")
            print(f"   📊 Avg detections/frame: {data_stats['avg_detections_per_frame']:.1f}")
            
            conf_stats = data_stats['confidence_stats']
            print(f"   🎯 Detection confidence: {conf_stats['mean']:.3f} ± {conf_stats['std']:.3f}")
        
        print(f"\n💡 REAL SIGLIP RECOMMENDATIONS:")
        print("   ✅ SigLIP model is working with real visual features")
        print("   🎯 Feature extraction from real player images successful")
        print("   🔧 Fine-tune clustering parameters (n_clusters, UMAP settings)")
        
        if perf['fps_estimate'] < 15:
            print("   ⚡ Consider GPU acceleration for better performance")
            print("   📦 Optimize batch size for your hardware")
        
        if balance_ratio < 0.7:
            print("   ⚖️ Team imbalance detected - adjust clustering algorithm")
        
        if quality['avg_confidence'] < 0.7:
            print("   🎨 Consider different feature extraction parameters")
            print("   📊 Experiment with UMAP dimensionality reduction settings")
        
        print(f"\n🚀 NEXT STEPS FOR REAL DEPLOYMENT:")
        print("   1. Test on multiple diverse video datasets")
        print("   2. Compare with manual ground truth annotations")
        print("   3. Implement temporal smoothing for stability")
        print("   4. Add jersey color analysis for validation")
        print("   5. Deploy in real-time processing pipeline")
        print("   6. Monitor performance on different camera angles")
        
    else:
        print("❌ No real SigLIP results available for evaluation")
        print("💡 This evaluation requires:")
        print("   - Real video data with player detections")
        print("   - Working SigLIP model with internet access")
        print("   - Sufficient computational resources")

# Print comprehensive real evaluation summary
if dashboard_stats and data_stats and dashboard_stats['method'] == 'SigLIP':
    print_real_evaluation_summary(dashboard_stats, data_stats)
    print("\n" + "=" * 70)
    print("🎉 Real SigLIP Team Assignment Evaluation Complete!")
    print("✅ All processing used real data and real models")
    print("=" * 70)
else:
    print("⚠️ Real evaluation incomplete or failed")
    print("💡 Check previous cells for real data loading and SigLIP processing errors")
    print("\n" + "=" * 70)
    print("❌ Real SigLIP Evaluation Failed")
    print("=" * 70)

# Diagnostic: Test SigLIP model loading and processor
print("🔍 Running SigLIP Diagnostics...")
print("=" * 50)

# Check local model path
local_model_path = project_root / "models" / "embed" / "team_assignment_model"
print(f"Local model path: {local_model_path}")
print(f"Path exists: {local_model_path.exists()}")

if local_model_path.exists():
    print(f"Model files:")
    for file in local_model_path.iterdir():
        print(f"  - {file.name}")

# Test transformers import
try:
    from transformers import AutoProcessor, SiglipVisionModel
    print("✅ Transformers imports successful")
except Exception as e:
    print(f"❌ Transformers import failed: {e}")

# Test direct model loading from local path
if local_model_path.exists():
    print("\n🧪 Testing direct model loading from local path...")
    try:
        # Test processor loading
        processor = AutoProcessor.from_pretrained(str(local_model_path))
        print("✅ AutoProcessor loaded from local path")
        
        # Test model loading
        model = SiglipVisionModel.from_pretrained(str(local_model_path))
        print("✅ SiglipVisionModel loaded from local path")
        
        # Test device transfer
        model = model.to(device)
        print(f"✅ Model transferred to {device}")
        
        # Test basic inference
        import torch
        from PIL import Image
        
        # Create a dummy image
        dummy_image = Image.new('RGB', (224, 224), color='red')
        inputs = processor(images=[dummy_image], return_tensors="pt").to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            embeddings = torch.mean(outputs.last_hidden_state, dim=1)
            print(f"✅ Inference test successful! Embedding shape: {embeddings.shape}")
        
        print("🎉 All local model tests passed!")
        
    except Exception as e:
        print(f"❌ Local model test failed: {e}")
        print(f"   Error type: {type(e).__name__}")
        import traceback
        print(f"   Full traceback:\n{traceback.format_exc()}")
else:
    print("⚠️  Cannot test local model - path does not exist")

# Test SigLIP processor class
print(f"\n🧪 Testing SigLIP Processor class...")
try:
    if local_model_path.exists():
        test_processor = SigLIPTeamAssignmentProcessor(
            model_path=str(local_model_path),
            device=device,
            batch_size=2,  # Small batch for testing
            n_clusters=2
        )
        print("✅ SigLIPTeamAssignmentProcessor instantiated successfully!")
        
        # Test attributes
        print(f"   - Device: {test_processor.device}")
        print(f"   - Batch size: {test_processor.batch_size}")
        print(f"   - Model path: {test_processor.model_path}")
        print(f"   - Is fitted: {test_processor._is_fitted}")
        
    else:
        print("⚠️  Cannot test processor - local model not available")
        
except Exception as e:
    print(f"❌ SigLIP processor test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    import traceback
    print(f"   Full traceback:\n{traceback.format_exc()}")

print("\n" + "=" * 50)
print("🏁 Diagnostic complete!")

# 🏈 SigLIP Team Assignment Processor Evaluation

**Visual evaluation and analysis of the SigLIPTeamAssignmentProcessor**

This notebook provides comprehensive evaluation of the SigLIP-based team assignment system with:
- 🎯 Performance benchmarking
- 📊 Visual analysis and clustering quality
- 🔄 Feature extraction evaluation
- 📈 Interactive dashboards

---

In [ ]:
# Environment Information
print("🔧 ENVIRONMENT INFORMATION")
print("=" * 50)

import sys
print(f"🐍 Python version: {sys.version}")
print(f"📁 Working directory: {project_root}")

# Check package availability
packages_to_check = ['torch', 'transformers', 'ultralytics', 'cv2', 'numpy', 'matplotlib', 'umap', 'sklearn']
print("\n📦 PACKAGE AVAILABILITY:")
for pkg in packages_to_check:
    try:
        if pkg == 'cv2':
            import cv2
            print(f"   ✅ OpenCV: {cv2.__version__}")
        elif pkg == 'torch':
            import torch
            print(f"   ✅ PyTorch: {torch.__version__}")
        elif pkg == 'transformers':
            import transformers
            print(f"   ✅ Transformers: {transformers.__version__}")
        elif pkg == 'ultralytics':
            import ultralytics
            print(f"   ✅ Ultralytics: {ultralytics.__version__}")
        elif pkg == 'numpy':
            import numpy
            print(f"   ✅ NumPy: {numpy.__version__}")
        elif pkg == 'matplotlib':
            import matplotlib
            print(f"   ✅ Matplotlib: {matplotlib.__version__}")
        elif pkg == 'umap':
            import umap
            print(f"   ✅ UMAP: {umap.__version__}")
        elif pkg == 'sklearn':
            import sklearn
            print(f"   ✅ Scikit-learn: {sklearn.__version__}")
    except ImportError:
        print(f"   ❌ {pkg}: Not available")

print(f"\n🎯 EVALUATION STATUS:")
if 'test_video_data' in locals() and test_video_data is not None:
    print(f"   📊 Data loaded: ✅ ({len(test_video_data.frames)} frames)")
else:
    print(f"   📊 Data loaded: ❌")

if 'siglip_processor' in locals() and siglip_processor is not None:
    print(f"   🤖 SigLIP processor: ✅")
else:
    print(f"   🤖 SigLIP processor: ❌")

print(f"   📈 Visualizations: ✅ (matplotlib only)")
print(f"   🎨 Dashboard: ✅")
print(f"   📝 Summary: ✅")

print("\n" + "=" * 50)
print("🎉 Notebook setup complete! Using matplotlib for all visualizations.")